<a href="https://colab.research.google.com/github/josenomberto/UTEC-CDIAV3-MCD8012/blob/main/laboratorioCalificado3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Importación de Librerias

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Configuración del dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando el dispositivo: {device}")

Utilizando el dispositivo: cpu


In [2]:
!pip install -q kaggle
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip

Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
100% 60.3M/60.3M [00:00<00:00, 156MB/s]



## 2. Transformaciones y Carga de Dataset

In [3]:
# Definición de las transformaciones necesarias para VGG16
vgg_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # VGG16 requiere dimensiones espaciales de 224x224
    transforms.ToTensor(),         # Convierte a tensor en rango [0.0, 1.0]
    transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x), # Convierte escala de grises a 3 canales RGB si es necesario
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # Medias de ImageNet
        std=[0.229, 0.224, 0.225]   # Desviaciones estándar de ImageNet
    )
])

# Reemplazar por las rutas locales correspondientes tras descomprimir fer2013 en Colab
# !pip install -q kaggle
# !kaggle datasets download -d msambare/fer2013
# !unzip -q fer2013.zip

train_dir = 'train'  # Ruta de la carpeta de entrenamiento original
test_dir = 'test'    # Ruta de la carpeta de prueba original

# Carga utilizando datasets.ImageFolder respetando la división original
train_dataset = datasets.ImageFolder(root=train_dir, transform=vgg_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=vgg_transforms)

# Definición de DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Imágenes de entrenamiento cargadas: {len(train_dataset)}")
print(f"Imágenes de prueba cargadas: {len(test_dataset)}")
print(f"Clases detectadas automáticamente: {train_dataset.classes}") # Debería mapear las 7 emociones básicas

Imágenes de entrenamiento cargadas: 28709
Imágenes de prueba cargadas: 7178
Clases detectadas automáticamente: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


## 3. Transfer Learning - Configuración del Modelo VGG16

In [4]:
# Carga del modelo VGG16 pre-entrenado con los pesos de ImageNet
vgg16 = models.vgg16(weights='IMAGENET1K_V1') # Equivalente a weights=models.VGG16_Weights.DEFAULT

# Congelación de todos los parámetros de las capas convolucionales (bloque de extracción de características)
for param in vgg16.features.parameters():
    param.requires_grad = False

# Modificación de la capa de salida del clasificador
# VGG16 tiene en su clasificador final una estructura secuencial donde la última capa lineal está en el índice [6]
in_features = vgg16.classifier[6].in_features
num_classes = len(train_dataset.classes) # El dataset FER-2013 posee exactamente 7 clases exclusivas

# Reemplazo por una nueva capa lineal adaptada a nuestro espacio de hipótesis
vgg16.classifier[6] = nn.Linear(in_features, num_classes)

# Traslado del modelo al hardware de aceleración (GPU/CPU)
vgg16 = vgg16.to(device)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 76.1MB/s]


## 4. Entrenamiento del Modelo

In [5]:
# Definición de la función de pérdida y el optimizador
criterion = nn.CrossEntropyLoss()
# Solo optimizamos los parámetros que tienen requires_grad = True (las del clasificador adaptado)
optimizer = optim.Adam(vgg16.classifier.parameters(), lr=0.001)

epochs = 5
train_losses = []
train_accuracies = []

print("Iniciando el proceso de ajuste del clasificador por 5 épocas...")
for epoch in range(epochs):
    vgg16.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = vgg16(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct / total) * 100

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)

    print(f"Época [{epoch+1}/{epochs}] -> Loss de Entrenamiento: {epoch_loss:.4f} | Precisión (Accuracy): {epoch_acc:.2f}%")

# Graficar la evolución del Entrenamiento (Función de costo e Historial de métricas)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, marker='o', color='red', label='Loss')
plt.title('Evolución de la Función de Costo')
plt.xlabel('Épocas')
plt.ylabel('Pérdida (Cross Entropy)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs+1), train_accuracies, marker='s', color='blue', label='Accuracy')
plt.title('Evolución de la Exactitud (Accuracy)')
plt.xlabel('Épocas')
plt.ylabel('Porcentaje (%)')
plt.grid(True)
plt.show()

Iniciando el proceso de ajuste del clasificador por 5 épocas...
Época [1/5] -> Loss de Entrenamiento: 1.5160 | Precisión (Accuracy): 42.87%


KeyboardInterrupt: 

## 5. Evaluación del Modelo

In [ ]:
vgg16.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = vgg16(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Generar reporte de clasificación analítico
print("\n--- REPORTE DE CLASIFICACIÓN (MÉTRICAS POR CLASE) ---")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

# Matriz de Confusión
cm = confusion_matrix(all_labels, all_preds)
print("--- MATRIZ DE CONFUSIÓN ---")
print(cm)